In [10]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../'))
sys.path.append(base_path)

import numpy as np
import matplotlib.pyplot as plt  
import pandas as pd
from tqdm import tqdm
import pandas as pd
from IPython.display import display

import project_images_loader as pil
import utils.util as util
import utils.functions as fc
import utils.nifti_functions as nfc
import utils.visualization as vis
from scipy import stats

np.set_printoptions(precision=4)

val_ids = ["08202"] + pil.get_matching_ids()[621:]


# Initial metrics

In [11]:
features_loader_tissue_based = fc.load_module_from_path('features_loader_tissue_based', os.path.join(base_path, 'statistics/patients/longitudinal/tissue_based/features_loader.py'))
initial_longitudinal_loader = features_loader_tissue_based.FeaturesLoader(3, val_ids)
initial_longitudinal_data = initial_longitudinal_loader.get_df(metrics_name=['dice'], remove_metrics=False, return_patient_id=True)

def get_initial_metric(metric):
    _val_ids = [int(i) for i in val_ids]
    if metric in ["ssim", "ncc", "psnr"]:
        _res = pd.read_csv(f'{base_path}/statistics/patients/longitudinal/whole_brain_based/features/whole_brains_stats.csv')
        _res = _res[_res['patient_id'].isin(_val_ids)]
        _res = _res[['patient_id', metric]]
    elif metric == "dice_csf":
        _res = initial_longitudinal_data[['patient_id', "1_dice"]]
    elif metric == "dice_gm":
        _res = initial_longitudinal_data[['patient_id', "2_dice"]]
    elif metric == "dice_wm":
        _res = initial_longitudinal_data[['patient_id', "3_dice"]]
    else:
        raise ValueError(f"Metric {metric} not found")
    return _res

def get_initial_mean(metric):
    return get_initial_metric(metric).mean()


In [12]:
def get_result_metric(approach_name, statistics_folder, metric, sufix_metric=''):
    _metric = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/{metric}{sufix_metric}.csv')
    return _metric

def get_mean_methods(approach_name, statistics_folder, metric):
    return get_result_metric(approach_name, statistics_folder, metric).mean().iloc[1:]

def get_mean_dices(approach_name, statistics_folder):
    _dice_csf = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/dice_csf.csv')
    _dice_gm = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/dice_gm.csv')
    _dice_wm = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/dice_wm.csv')
    _mean_csf = _dice_csf.mean().iloc[1:]
    _mean_gm = _dice_gm.mean().iloc[1:]
    _mean_wm = _dice_wm.mean().iloc[1:]

    means = pd.DataFrame({
        'mean_csf': _mean_csf,
        'mean_gm': _mean_gm,
        'mean_wm': _mean_wm
    })
    return means

from sklearn.preprocessing import MinMaxScaler
def min_max_scale(df):
    scaler = MinMaxScaler()
    return pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)

def create_method_table(method, statistics_folder='statistics', sufix_metric=''):
    # metrics = ['ssim', 'ncc', 'psnr', 'dice_csf', 'dice_gm', 'dice_wm']
    metrics = ['ssim', 'dice_csf', 'dice_gm', 'dice_wm']
    # mult = [100, 100, 1, 100, 100, 100]
    mult = [100, 100, 100, 100]

    res_dict = {}
    res_means_dict = {}
    res_sem_dict = {}
    res_t_test_dict = {}
    res_p_value_dict = {}
    columns = None
    for i, metric in enumerate(metrics):
        _initial_res = get_initial_metric(metric) * mult[i]
        if method in [None, 'initial']:
            res_dict[metric] = _initial_res
            columns = ['initial']
            # _initial_res = min_max_scale(_initial_res)  # Normalizar los datos iniciales

        else:
            res_dict[metric] = get_result_metric(method, statistics_folder, metric, sufix_metric=sufix_metric) * mult[i]
            # res_dict[metric] = min_max_scale(res_dict[metric])  # Normalizar los resultados

            columns = res_dict[metric].columns[1:] 

        t_stat_A, p_val_A = stats.ttest_rel(res_dict[metric].iloc[:,1:], _initial_res.values[:,1:])
        res_t_test_dict[metric] = t_stat_A
        res_p_value_dict[metric] = p_val_A
        res_means_dict[metric] = res_dict[metric].iloc[:,1:].mean()
        # res_sem_dict[metric] = stats.sem(res_dict[metric].iloc[:,1:], axis=0)
        res_sem_dict[metric] = res_dict[metric].iloc[:,1:].std()

    # create np table
    res_table = np.zeros((len(columns), len(metrics), 4))
    for i, metric in enumerate(metrics):
        res_table[:, i, 0] = res_means_dict[metric]
        res_table[:, i, 1] = res_sem_dict[metric]
        res_table[:, i, 2] = res_t_test_dict[metric]
        res_table[:, i, 3] = res_p_value_dict[metric]

    # create pd table
    df_mean = pd.DataFrame(res_table[...,0], columns=metrics, index=columns)
    df_sem = pd.DataFrame(res_table[...,1], columns=metrics, index=columns)
    df_t_test = pd.DataFrame(res_table[...,2], columns=metrics, index=columns)
    df_p_value = pd.DataFrame(res_table[...,3], columns=metrics, index=columns)
    return df_mean, df_sem, df_t_test, df_p_value






# Atlas based methods

In [13]:
pd.options.display.float_format = '{:.1f}'.format

atlas_based_methods = ['initial', 'dfield_mean', 'dfield_similar_images', 'dfield_similar_images_registered', 'dfield_similar_tissues', 'dfield_similar_tissues_v2', 'dfield_similar_patches', 'dfield_similar_patches']
# atlas_based_methods = ['initial', 'dfield_similar_images', 'dfield_similar_tissues', 'dfield_similar_patches']

postprocessing_increment = {"ssim": 0.275, "dice_csf": 1.71, "dice_gm": 1.931, "dice_wm": 1.012}
for i, method in enumerate(atlas_based_methods):
    print(method)
    statistics_folder = 'statistics_postprocessed' if i == len(atlas_based_methods)-1 else 'statistics'
    df_mean, df_sem, df_t_test, df_p_value = create_method_table(method, statistics_folder=statistics_folder)

    # if i < len(atlas_based_methods)-1: 
    #     for col, increment in postprocessing_increment.items():
    #         df_mean[col] += increment

    display(df_mean )
    # display(df_sem)
    # display(df_t_test)
    display(df_p_value)
# csf = 1.71
# gm = 1.931
# wm = 1.012
# SSIM = 0.275

initial


,ssim,dice_csf,dice_gm,dice_wm
initial,94.6,84.3,80.5,90.0


,ssim,dice_csf,dice_gm,dice_wm
initial,NaN,NaN,NaN,NaN


dfield_mean


,ssim,dice_csf,dice_gm,dice_wm
10,95.1,88.3,80.8,90.4
100,95.3,88.9,81.7,90.9
200,95.3,88.8,81.8,90.9
300,95.3,88.8,81.8,90.9
620,95.4,88.8,81.8,90.9


...

,ssim,dice_csf,dice_gm,dice_wm
3_10,95.5,90.5,82.2,91.2
3_100,95.5,90.1,82.2,91.2
96_10,95.5,90.5,82.2,91.2
96_100,95.5,90.2,82.2,91.2


,ssim,dice_csf,dice_gm,dice_wm
3_10,0.0,0.0,0.0,0.0
3_100,0.0,0.0,0.0,0.0
96_10,0.0,0.0,0.0,0.0
96_100,0.0,0.0,0.0,0.0


dfield_similar_patches


,ssim,dice_csf,dice_gm,dice_wm
96_10,95.8,92.2,84.1,92.2


,ssim,dice_csf,dice_gm,dice_wm
96_10,0.0,0.0,0.0,0.0


# Deep learning methods

In [14]:
pd.options.display.float_format = '{:.3f}'.format

# atlas_based_methods = ['initial', 'MGAN_paper_16', 'MPGAN_paper_16_seg_loss', 'MPGAN_paper_16_seg_loss', 'StarGAN_V2_sagital', 'diffusion_model']
# atlas_based_methods = ['initial', 'MPGAN_paper_16_seg_loss', 'MPGAN_paper_16', 'MGAN_paper_16', 'StarGAN_V2_sagital', 'diffusion_model']
atlas_based_methods = ['initial', 'MPGAN_paper_16', 'MPGAN_paper_16_seg_loss', 'MGAN_paper_16', 'StarGAN_V2_sagital']
# sufin_metric = ['', '_RAW', '',  '', '', '']
sufin_metric = ['', '', '_RAW',  '', '']
for i, method in enumerate(atlas_based_methods):
    print(method)
    df_mean, df_sem, df_t_test, df_p_value = create_method_table(method, statistics_folder='statistics_postprocessed', sufix_metric=sufin_metric[i])
    display(df_mean )
    # display(df_sem)
    # display(df_t_test)
    display(df_p_value)


#           ssim	ncc	psnr	dice_csf	dice_gm	dice_wm
# initial	94.64	96.68	25.64	84.27	80.51	90.01


#       ssim	ncc	psnr	dice_csf	dice_gm	dice_wm
# RES	92.42	94.67	23.87	88.30	74.65	86.97

initial


,ssim,dice_csf,dice_gm,dice_wm
initial,94.636,84.266,80.506,90.006


,ssim,dice_csf,dice_gm,dice_wm
initial,NaN,NaN,NaN,NaN


MPGAN_paper_16


,ssim,dice_csf,dice_gm,dice_wm
28,92.055,88.146,72.172,86.220


...

,ssim,dice_csf,dice_gm,dice_wm
46,92.583,89.223,72.720,87.110


,ssim,dice_csf,dice_gm,dice_wm
46,0.000,0.000,0.000,0.000


StarGAN_V2_sagital


,ssim,dice_csf,dice_gm,dice_wm
BEST,92.750,82.421,75.227,87.109


,ssim,dice_csf,dice_gm,dice_wm
BEST,0.000,0.101,0.000,0.000


# Best methods

In [15]:
pd.options.display.float_format = '{:.2f}'.format

atlas_based_methods = ['initial', 'dfield_similar_patches', 'MPGAN_paper_16_seg_loss']
sufin_metric = ['', '', '_RAW']
for i, method in enumerate(atlas_based_methods):
    print(method)
    df_mean, df_sem, df_t_test, df_p_value = create_method_table(method, statistics_folder='statistics_postprocessed', sufix_metric=sufin_metric[i])
    display(df_mean)
    display(df_sem)

initial


,ssim,dice_csf,dice_gm,dice_wm
initial,94.64,84.27,80.51,90.01


,ssim,dice_csf,dice_gm,dice_wm
initial,0.97,4.84,2.59,1.85


dfield_similar_patches


,ssim,dice_csf,dice_gm,dice_wm
96_10,95.80,92.22,84.14,92.23


,ssim,dice_csf,dice_gm,dice_wm
96_10,0.74,2.75,2.37,1.46


MPGAN_paper_16_seg_loss


,ssim,dice_csf,dice_gm,dice_wm
52,94.85,90.68,81.38,91.01


,ssim,dice_csf,dice_gm,dice_wm
52,0.72,3.22,2.17,1.29


# t-statistics

In [16]:
# ssim_initial = get_initial_metric('ssim')
# ssim_dfield_mean = get_result_metric('dfield_mean', 'statistics', 'ssim')
# ssim_dfield_similar_images = get_result_metric('dfield_similar_images', 'statistics', 'ssim')
# ssim_dfield_similar_images_registered = get_result_metric('dfield_similar_images_registered', 'statistics', 'ssim')
# ssim_dfield_similar_tissues = get_result_metric('dfield_similar_tissues', 'statistics', 'ssim')
# ssim_dfield_similar_tissues2 = get_result_metric('dfield_similar_tissues_v2', 'statistics', 'ssim')
# ssim_dfield_similar_patches = get_result_metric('dfield_similar_patches', 'statistics', 'ssim')


# def show_t_statistics(ssim_initial, ssim_res):

#     for i in range(0, ssim_res.shape[1]-1):
#         t_stat_A, p_val_A = stats.ttest_rel(ssim_res.values[:, i+1], ssim_initial.values)
#         print(f'{ssim_res.columns[i+1]}: {t_stat_A} - {p_val_A}')

# print('mean_ssim_dfield_mean:')
# show_t_statistics(ssim_initial, ssim_dfield_mean)
# print('\nmean_ssim_dfield_similar_images:')
# show_t_statistics(ssim_initial, ssim_dfield_similar_images)
# print('\nmean_ssim_dfield_similar_images_registered:')
# show_t_statistics(ssim_initial, ssim_dfield_similar_images_registered)
# print('\nmean_ssim_dfield_similar_tissues:')
# show_t_statistics(ssim_initial, ssim_dfield_similar_tissues)
# print('\nmean_ssim_dfield_similar_tissues:')
# show_t_statistics(ssim_initial, ssim_dfield_similar_tissues2)
# print('\nmean_ssim_dfield_similar_patches:')
# show_t_statistics(ssim_initial, ssim_dfield_similar_patches)




# Compare best methods

In [17]:

def get_mean(approach_name, statistics_folder, metric):
    _metric = pd.read_csv(f'{current_path}/{approach_name}/images/{statistics_folder}/{metric}.csv')
    return _metric.mean().values[1]

approach_name_list = ["Initial", "dfield_similar_patches", "MPGAN_paper_16_seg_loss", "MPGAN_paper_16", "MGAN_paper_16", "StarGAN_V2_sagital"]
metric_list = ["ssim", "dice_csf", "dice_gm", "dice_wm"]
# metric_list = ["ssim"]
for metric in metric_list:
    print(f'{metric} mean'.upper())
    for approach_name in approach_name_list:
            if approach_name == "Initial":
                print(f'{approach_name}:\t\t\t', get_initial_mean(metric))
            elif approach_name == "MPGAN_paper_16_seg_loss":
                print(f'{approach_name} RAW:\t', get_mean(approach_name, "statistics_postprocessed", metric + "_raw"))
                print(f'{approach_name} POST:\t', get_mean(approach_name, "statistics_postprocessed", metric + "_post"))
            elif approach_name == "dfield_similar_patches":
                print(f'{approach_name}:\t\t', get_mean(approach_name, "statistics_postprocessed", metric))
            elif approach_name == "MPGAN_paper_16":
                print(f'{approach_name}:\t\t\t', get_mean(approach_name, "statistics_postprocessed", metric))
            elif approach_name == "MGAN_paper_16":
                print(f'{approach_name}:\t\t\t', get_mean(approach_name, "statistics_postprocessed", metric))
            elif approach_name == "StarGAN_V2_sagital":
                print(f'{approach_name}:\t\t', get_mean(approach_name, "statistics_postprocessed", metric))
    print('')


SSIM MEAN
Initial:			 patient_id   15234.72
ssim             0.95
dtype: float64
dfield_similar_patches:		 0.9580158316164026
MPGAN_paper_16_seg_loss RAW:	 0.9485469389841606
MPGAN_paper_16_seg_loss POST:	 0.9272257148807934
MPGAN_paper_16:			 0.9205466891584817
MGAN_paper_16:			 0.9258335220137155
StarGAN_V2_sagital:		 0.9275009487400047

DICE_CSF MEAN
Initial:			 patient_id   15234.72
1_dice           0.84
dtype: float32
dfield_similar_patches:		 0.922201029920219
MPGAN_paper_16_seg_loss RAW:	 0.906797215690076
MPGAN_paper_16_seg_loss POST:	 0.9009654682925385
MPGAN_paper_16:			 0.8814639295018047
MGAN_paper_16:			 0.8922287746772473
StarGAN_V2_sagital:		 0.8242091928120957

DICE_GM MEAN
Initial:			 patient_id   15234.72
2_dice           0.81
dtype: float32
dfield_similar_patches:		 0.8413915756618774
MPGAN_paper_16_seg_loss RAW:	 0.8137642209791004
MPGAN_paper_16_seg_loss POST:	 0.7496597890349561
MPGAN_paper_16:			 0.7217219685105815
MGAN_paper_16:			 0.7271998346811397
StarGAN_V2_

### Brain parenchymal fraction

In [18]:
val_images = ["08202"] + pil.get_matching_ids()[621:]

bpf_orig = pd.read_csv(f'C:/Users/Agustin/Master/Master/S4/code/statistics/patients/cross_sectional/whole_brain/features/whole_brains_stats.csv')
bpf_orig = bpf_orig[bpf_orig["patient_id"].isin([int(x) for x in val_images])]
bpf_orig = bpf_orig.reset_index(drop=True)
bpf_dfield_similar = pd.read_csv(f'{current_path}/dfield_similar_patches/images/statistics_postprocessed/bpf.csv')
bpf_MPGAN_raw = pd.read_csv(f'{current_path}/MPGAN_paper_16_seg_loss/images/statistics_postprocessed/bpf_raw.csv')
bpf_starGAN = pd.read_csv(f'{current_path}/StarGAN_V2_sagital/images/statistics_postprocessed/bpf.csv')

bpf_orig["bpf_sim_patches"] = bpf_dfield_similar["96_10"]
bpf_orig["bpf_MPGAN_raw"] = bpf_MPGAN_raw["52"]
bpf_orig["bpf_starGAN"] = bpf_starGAN["BEST"]
bpf_orig["bpf_sim_patches_dif"] = (bpf_orig["bpf_hunt3"] - bpf_orig["bpf_sim_patches"])/(bpf_orig["bpf_hunt3"] - bpf_orig["bpf_hunt4"])
bpf_orig["bpf_MPGAN_raw_dif"] = (bpf_orig["bpf_hunt3"] - bpf_orig["bpf_MPGAN_raw"])/(bpf_orig["bpf_hunt3"] - bpf_orig["bpf_hunt4"])
bpf_orig["bpf_starGAN_dif"] = (bpf_orig["bpf_hunt3"] - bpf_orig["bpf_starGAN"])/(bpf_orig["bpf_hunt3"] - bpf_orig["bpf_hunt4"])
display(bpf_orig.head(20))
print("BPF MEAN:\n", bpf_orig.mean()[1:])
print("\nBPF MAX:\n", bpf_orig.max()[1:])


,patient_id,bpf_hunt3,bpf_hunt4,bpf_sim_patches,bpf_MPGAN_raw,bpf_starGAN,bpf_sim_patches_dif,bpf_MPGAN_raw_dif,bpf_starGAN_dif
0,8202.00,0.79,0.76,0.78,0.75,0.73,0.48,1.53,2.74
1,14098.00,0.74,0.71,0.73,0.74,0.70,0.38,0.16,1.51
2,14104.00,0.75,0.72,0.73,0.72,0.69,0.79,0.84,2.17
3,14128.00,0.76,0.74,0.75,0.75,0.72,0.40,0.50,1.77
4,14135.00,0.78,0.76,0.77,0.76,0.74,0.43,1.02,1.69
5,14159.00,0.75,0.73,0.73,0.73,0.70,1.02,1.00,3.82
6,14180.00,0.74,0.71,0.73,0.73,0.70,0.47,0.12,1.33
7,14197.00,0.73,0.70,0.72,0.73,0.70,0.32,-0.01,0.97
8,14203.00,0.76,0.73,0.74,0.74,0.71,0.65,0.57,2.01
9,14241.00,0.78,0.75,0.77,0.76,0.73,0.31,0.87,1.83


BPF MEAN:
 bpf_hunt3             0.75
bpf_hunt4             0.72
bpf_sim_patches       0.74
bpf_MPGAN_raw         0.74
bpf_starGAN           0.71
bpf_sim_patches_dif   0.53
bpf_MPGAN_raw_dif     0.49
bpf_starGAN_dif       1.74
dtype: float64

BPF MAX:
 bpf_hunt3             0.79
bpf_hunt4             0.76
bpf_sim_patches       0.78
bpf_MPGAN_raw         0.76
bpf_starGAN           0.74
bpf_sim_patches_dif   1.08
bpf_MPGAN_raw_dif     1.53
bpf_starGAN_dif       3.82
dtype: float64
